In [34]:
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset


def sparse_bow_collate(batch):
    xs, ys = zip(*batch)
    x = sp.vstack(xs).toarray().astype(np.float32)
    return torch.from_numpy(x), torch.stack(ys)


class SparseBowDataset(Dataset):
    def __init__(self, X_sparse, y_1based):
        self.X = X_sparse
        self.y = torch.from_numpy(y_1based.astype(np.int64) - 1)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class BowMLP(nn.Module):
    def __init__(self, in_dim=10000, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 1024)
        self.drop1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(1024, 256)
        self.drop2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.drop1(x)
        x = F.gelu(self.fc2(x))
        x = self.drop2(x)
        return self.fc3(x)


In [ ]:
import random
import re
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

nltk.download("stopwords", quiet=True)

DATA_DIR = Path("../10_categories_of_yahoo_answers_for_nlp_tasks_csv")
TRAIN_PATH = DATA_DIR / "train_mini.csv"
TEST_PATH = DATA_DIR / "test.csv"

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()


def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def preprocess_df(df):
    title = df["question_title"].fillna("").astype(str)
    content = df["question_content"].fillna("").astype(str)
    question = (title + " " + content).str.strip()
    return question.apply(normalize_text)


def tokenize_for_vectorizer(text):
    text = text.lower().strip()
    tokens = re.findall(r"\b\w+\b", text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return tokens


train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
train_df["text"] = preprocess_df(train_df)
test_df["text"] = preprocess_df(test_df)
train_df["class_index"] = pd.to_numeric(train_df["class_index"], errors="coerce").astype("Int64")
test_df["class_index"] = pd.to_numeric(test_df["class_index"], errors="coerce").astype("Int64")
train_df = train_df.dropna(subset=["class_index"])
test_df = test_df.dropna(subset=["class_index"])

X_raw = train_df["text"].values
y = train_df["class_index"].astype(int).values
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)
X_test_raw = test_df["text"].values
y_test = test_df["class_index"].astype(int).values

tfidf = TfidfVectorizer(
    tokenizer=tokenize_for_vectorizer,
    lowercase=False,
    max_features=50000,
)
X_train = tfidf.fit_transform(X_train_raw)
X_val = tfidf.transform(X_val_raw)
X_test = tfidf.transform(X_test_raw)
print(X_train.shape, X_val.shape, X_test.shape)

/Users/tyleryue/Desktop/4NL3/project/project-4nl3/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


(160000, 50000) (40000, 50000) (59999, 50000)


In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("mps")
batch_size = 256
epochs = 10

train_loader = DataLoader(
    SparseBowDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True,
    collate_fn=sparse_bow_collate,
)

val_loader = DataLoader(
    SparseBowDataset(X_val, y_val),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=sparse_bow_collate,
)

import torch.nn.functional as F

class NeuralNetwork(nn.Module):
    def __init__(self, in_dim=50000, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 512)
        self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(512, 64)
        self.drop2 = nn.Dropout(0.4)
        self.fc4 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        #x = self.bn1(x)
        x = F.relu(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.drop2(x)
        # x = self.fc3(x)
        # x = F.leaky_relu(x)
        # x = self.drop3(x)
        x = self.fc4(x)
        return x

model = NeuralNetwork(in_dim=X_train.shape[1], num_classes=10).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-4)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss_fn(model(xb), yb).backward()
        opt.step()
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    print("epoch", epoch, "val_acc", correct / total)

epoch 0 val_acc 0.59035
epoch 1 val_acc 0.660025
epoch 2 val_acc 0.67475
epoch 3 val_acc 0.68275
epoch 4 val_acc 0.686325
epoch 5 val_acc 0.68725
epoch 6 val_acc 0.687725
epoch 7 val_acc 0.6867
epoch 8 val_acc 0.684875
epoch 9 val_acc 0.68355


In [30]:
test_loader = DataLoader(
    SparseBowDataset(X_test, y_test),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=sparse_bow_collate,
)

model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds.append(model(xb.to(device)).argmax(1).cpu().numpy())
y_pred = np.concatenate(preds) + 1

print("test acc", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

out_dir = Path("results/pytorch_mlp")
out_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"class_index": y_pred}).to_csv(out_dir / "testing_label.csv", index=False)

test acc 0.6844114068567809
              precision    recall  f1-score   support

           1       0.58      0.57      0.57      6000
           2       0.66      0.69      0.67      6000
           3       0.71      0.77      0.74      6000
           4       0.55      0.48      0.51      6000
           5       0.81      0.83      0.82      6000
           6       0.86      0.85      0.85      6000
           7       0.56      0.48      0.51      6000
           8       0.68      0.67      0.68      6000
           9       0.68      0.76      0.72      5999
          10       0.72      0.75      0.74      6000

    accuracy                           0.68     59999
   macro avg       0.68      0.68      0.68     59999
weighted avg       0.68      0.68      0.68     59999



In [31]:
from sklearn.metrics import confusion_matrix
import pandas as pd

label_names = [
    "Society & Culture",
    "Science & Mathematics",
    "Health",
    "Education & Reference",
    "Computers & Internet",
    "Sports",
    "Business & Finance",
    "Entertainment & Music",
    "Family & Relationships",
    "Politics & Government"
]

# since your labels are 1 to 10
labels = list(range(1, 11))

cm = confusion_matrix(y_test, y_pred, labels=labels)

cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
print(cm_df)

                        Society & Culture  Science & Mathematics  Health  \
Society & Culture                    3398                    191     239   
Science & Mathematics                 189                   4154     402   
Health                                169                    303    4613   
Education & Reference                 461                    850     239   
Computers & Internet                   44                     99      32   
Sports                                 69                    105     129   
Business & Finance                    414                    310     298   
Entertainment & Music                 364                    164     157   
Family & Relationships                371                     42     276   
Politics & Government                 347                     91      91   

                        Education & Reference  Computers & Internet  Sports  \
Society & Culture                         423                    82      81   
Scien